In [ ]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Thiết lập hiển thị
pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

In [ ]:
# ==========================================
# 2. SET PROJECT PATH
# ==========================================

# Xác định thư mục gốc của dự án
PROJECT_DIR = Path.cwd().parent

# Thư mục lưu kết quả
RESULTS_DIR = PROJECT_DIR / "results"

# Kiểm tra đường dẫn
print("Project directory:", PROJECT_DIR)
print("Results directory:", RESULTS_DIR)

# Kiểm tra thư mục results có tồn tại không
print("Results exists:", RESULTS_DIR.exists())

In [ ]:
# ==========================================
# 3. LOAD MACHINE LEARNING RESULTS
# ==========================================

# Đọc kết quả 2 mô hình Machine Learning
ml_results = pd.read_csv(
    RESULTS_DIR / "ml_model_comparison.csv"
)

# Hiển thị kết quả
display(ml_results)

In [ ]:
# ==========================================
# 4. LOAD DEEP LEARNING RESULTS
# ==========================================

# Đọc kết quả MLP từ Stage 6
mlp_results = pd.read_csv(
    RESULTS_DIR / "mlp_model_results.csv"
)

# Hiển thị kết quả
display(mlp_results)

In [ ]:
# ==========================================
# 5. NORMALIZE COLUMN NAMES
# ==========================================

# Hàm chuẩn hóa tên cột để dễ xử lý
def normalize_result_columns(data):

    # Sao chép dataframe để không làm thay đổi dữ liệu gốc
    data = data.copy()

    # Loại bỏ khoảng trắng đầu/cuối tên cột
    data.columns = data.columns.str.strip()

    # Chuẩn hóa các tên metric có thể khác nhau
    rename_mapping = {
        "Model Name": "Model",
        "F1": "F1-score",
        "F1 Score": "F1-score",
        "PR-AUC Score": "PR-AUC"
    }

    # Đổi tên cột nếu có trong mapping
    data = data.rename(columns=rename_mapping)

    return data


# Chuẩn hóa Machine Learning
ml_results = normalize_result_columns(ml_results)

# Chuẩn hóa Deep Learning
mlp_results = normalize_result_columns(mlp_results)

# Kiểm tra tên cột
print("ML columns:", ml_results.columns.tolist())
print("MLP columns:", mlp_results.columns.tolist())

In [ ]:
# ==========================================
# 6. COMBINE ALL MODEL RESULTS
# ==========================================

# Kết hợp Machine Learning và Deep Learning
all_results = pd.concat(
    [ml_results, mlp_results],
    ignore_index=True
)

# Chỉ lấy các metrics cần so sánh
metric_columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "PR-AUC"
]

# Kiểm tra xem có thiếu cột nào không
missing_columns = [
    col for col in metric_columns
    if col not in all_results.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in model results: {missing_columns}"
    )

# Lọc và sắp xếp kết quả
comparison_df = all_results[metric_columns].copy()

# Làm tròn 4 chữ số thập phân
comparison_df = comparison_df.round(4)

# Hiển thị bảng
display(comparison_df)

In [ ]:
# ==========================================
# 7. SAVE MODEL COMPARISON
# ==========================================

# Lưu bảng so sánh 3 mô hình
comparison_df.to_csv(
    RESULTS_DIR / "all_model_comparison.csv",
    index=False
)

print("Comparison table saved!")
print("File: results/all_model_comparison.csv")

In [ ]:
# ==========================================
# 8. BEST MODEL FOR EACH METRIC
# ==========================================

# Những metric dùng để so sánh
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "PR-AUC"
]

# Tìm mô hình có kết quả cao nhất ở từng metric
best_models = []

for metric in metrics:

    # Vị trí mô hình có metric cao nhất
    best_index = comparison_df[metric].idxmax()

    # Lấy tên mô hình
    best_model_name = comparison_df.loc[
        best_index, "Model"
    ]

    # Lấy giá trị metric
    best_value = comparison_df.loc[
        best_index, metric
    ]

    # Lưu kết quả
    best_models.append({
        "Metric": metric,
        "Best Model": best_model_name,
        "Score": best_value
    })

# Chuyển thành dataframe
best_models_df = pd.DataFrame(best_models)

display(best_models_df)

In [ ]:
# ==========================================
# 9. MODEL METRICS COMPARISON
# ==========================================

# Chuyển bảng từ wide format sang long format
comparison_long = comparison_df.melt(
    id_vars="Model",
    value_vars=metrics,
    var_name="Metric",
    value_name="Score"
)

# Thiết lập kích thước biểu đồ
plt.figure(figsize=(15, 7))

# Vẽ grouped bar chart
sns.barplot(
    data=comparison_long,
    x="Metric",
    y="Score",
    hue="Model"
)

# Thiết lập tiêu đề và nhãn
plt.title(
    "Performance Comparison of Machine Learning and Deep Learning Models",
    fontsize=14
)

plt.xlabel("Evaluation Metrics")
plt.ylabel("Score")
plt.ylim(0, 1)

plt.xticks(rotation=0)
plt.legend(title="Model")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 10. RECALL COMPARISON
# ==========================================

plt.figure(figsize=(8, 5))

sns.barplot(
    data=comparison_df,
    x="Model",
    y="Recall"
)

plt.title("Recall Comparison Between Models")
plt.xlabel("Model")
plt.ylabel("Recall")
plt.ylim(0, 1)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 11. F1-SCORE COMPARISON
# ==========================================

plt.figure(figsize=(8, 5))

sns.barplot(
    data=comparison_df,
    x="Model",
    y="F1-score"
)

plt.title("F1-score Comparison Between Models")
plt.xlabel("Model")
plt.ylabel("F1-score")
plt.ylim(0, 1)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 12. IMPORT SHAP
# ==========================================

import shap

print("SHAP version:", shap.__version__)

In [ ]:
# ==========================================
# 13. LOAD CATBOOST MODEL
# ==========================================

from catboost import CatBoostClassifier

# Khởi tạo CatBoost
catboost_model = CatBoostClassifier()

# Load model đã huấn luyện
catboost_model.load_model(
    PROJECT_DIR / "models" / "catboost_model.cbm"
)

print("CatBoost model loaded successfully!")

In [ ]:
# ==========================================
# 14. LOAD DATA FOR SHAP ANALYSIS
# ==========================================

# Load dataset gốc
df_shap = pd.read_csv(
    PROJECT_DIR
    / "dataset"
    / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

# Loại bỏ các cột không sử dụng khi training CatBoost
drop_columns = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

X_shap = df_shap.drop(
    columns=drop_columns + ["Attrition"]
)

# Chuyển categorical về string như khi train CatBoost
categorical_features_shap = X_shap.select_dtypes(
    include=["object"]
).columns.tolist()

for col in categorical_features_shap:
    X_shap[col] = X_shap[col].astype(str)

print("SHAP data shape:", X_shap.shape)

display(X_shap.head())

In [ ]:
# ==========================================
# 15. SAMPLE DATA FOR SHAP
# ==========================================

# Lấy ngẫu nhiên tối đa 300 dòng để giải thích
X_shap_sample = X_shap.sample(
    n=min(300, len(X_shap)),
    random_state=42
)

print("SHAP sample shape:", X_shap_sample.shape)

In [ ]:
# ==========================================
# 16. CALCULATE SHAP VALUES
# ==========================================

# Tạo TreeExplainer cho CatBoost
shap_explainer = shap.TreeExplainer(
    catboost_model
)

# Tính SHAP values cho mẫu dữ liệu
shap_values = shap_explainer(
    X_shap_sample
)

print("SHAP values calculated successfully!")

In [ ]:
# ==========================================
# 17. SHAP SUMMARY PLOT
# ==========================================

# Hiển thị mức độ ảnh hưởng của các feature
shap.plots.beeswarm(
    shap_values,
    max_display=20
)

plt.show()

In [ ]:
# ==========================================
# 18. SHAP FEATURE IMPORTANCE
# ==========================================

# Tính mean absolute SHAP value của từng feature
shap_importance = np.abs(
    shap_values.values
).mean(axis=0)

# Tạo dataframe feature importance
shap_importance_df = pd.DataFrame({
    "Feature": X_shap_sample.columns,
    "Mean_Absolute_SHAP": shap_importance
})

# Sắp xếp từ cao xuống thấp
shap_importance_df = shap_importance_df.sort_values(
    by="Mean_Absolute_SHAP",
    ascending=False
)

# Hiển thị Top 20 features
display(shap_importance_df.head(20))

In [ ]:
# ==========================================
# 19. SHAP BAR PLOT
# ==========================================

# Vẽ 20 feature quan trọng nhất
shap.plots.bar(
    shap_values,
    max_display=20
)

plt.show()

In [ ]:
# ==========================================
# 20. LOCAL EXPLANATION
# ==========================================

# Lấy một mẫu cụ thể trong tập SHAP
sample_index = 0

# Hiển thị đặc trưng của mẫu
display(X_shap_sample.iloc[[sample_index]])

# Hiển thị dự đoán của CatBoost cho mẫu này
sample_prediction = catboost_model.predict_proba(
    X_shap_sample.iloc[[sample_index]]
)[0][1]

print(
    f"Predicted probability of attrition: "
    f"{sample_prediction:.4f}"
)

In [ ]:
# ==========================================
# 21. SHAP WATERFALL PLOT
# ==========================================

# Giải thích một mẫu cụ thể
shap.plots.waterfall(
    shap_values[sample_index],
    max_display=15
)

plt.show()

In [ ]:
# ==========================================
# 22. LOAD PREDICTION FILES
# ==========================================

# Logistic Regression và CatBoost predictions
ml_predictions = pd.read_csv(
    RESULTS_DIR / "ml_predictions.csv"
)

# MLP predictions
mlp_predictions = pd.read_csv(
    RESULTS_DIR / "mlp_predictions.csv"
)

# Kiểm tra các cột
print("ML prediction columns:")
print(ml_predictions.columns.tolist())

print("\nMLP prediction columns:")
print(mlp_predictions.columns.tolist())

display(mlp_predictions.head())

In [ ]:
# ==========================================
# 23. COMBINED ROC CURVE
# ==========================================

from sklearn.metrics import roc_curve, auc

# Lấy nhãn thực tế
y_true = mlp_predictions["Actual"].values

# Lấy xác suất dự đoán từ 3 mô hình
logistic_prob = ml_predictions["Logistic_Probability"].values
catboost_prob = ml_predictions["CatBoost_Probability"].values
mlp_prob = mlp_predictions["Probability_Attrition"].values

# ROC Logistic Regression
fpr_log, tpr_log, _ = roc_curve(y_true, logistic_prob)
auc_log = auc(fpr_log, tpr_log)

# ROC CatBoost
fpr_cat, tpr_cat, _ = roc_curve(y_true, catboost_prob)
auc_cat = auc(fpr_cat, tpr_cat)

# ROC MLP
fpr_mlp, tpr_mlp, _ = roc_curve(y_true, mlp_prob)
auc_mlp = auc(fpr_mlp, tpr_mlp)

# Vẽ biểu đồ
plt.figure(figsize=(8, 6))

plt.plot(
    fpr_log,
    tpr_log,
    label=f"Logistic Regression (AUC={auc_log:.3f})"
)

plt.plot(
    fpr_cat,
    tpr_cat,
    label=f"CatBoost (AUC={auc_cat:.3f})"
)

plt.plot(
    fpr_mlp,
    tpr_mlp,
    label=f"MLP (AUC={auc_mlp:.3f})"
)

# Đường baseline random classifier
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 24. COMBINED PRECISION-RECALL CURVE
# ==========================================

from sklearn.metrics import precision_recall_curve

# Tính PR cho Logistic Regression
precision_log, recall_log, _ = precision_recall_curve(
    y_true,
    logistic_prob
)

# Tính PR cho CatBoost
precision_cat, recall_cat, _ = precision_recall_curve(
    y_true,
    catboost_prob
)

# Tính PR cho MLP
precision_mlp, recall_mlp, _ = precision_recall_curve(
    y_true,
    mlp_prob
)

# Baseline = tỷ lệ positive trong test
baseline = y_true.mean()

# Vẽ biểu đồ
plt.figure(figsize=(8, 6))

plt.plot(
    recall_log,
    precision_log,
    label=f"Logistic Regression (AP={comparison_df.loc[comparison_df['Model"].eq('Logistic Regression'), 'PR-AUC'].iloc[0]:.3f})"
)

plt.plot(
    recall_cat,
    precision_cat,
    label=f"CatBoost (AP={comparison_df.loc[comparison_df['Model"].eq('CatBoost'), 'PR-AUC'].iloc[0]:.3f})"
)

plt.plot(
    recall_mlp,
    precision_mlp,
    label=f"MLP (AP={comparison_df.loc[comparison_df['Model"].eq('MLP Neural Network'), 'PR-AUC'].iloc[0]:.3f})"
)

# Baseline
plt.axhline(
    y=baseline,
    linestyle="--",
    label=f"Baseline = {baseline:.3f}"
)

plt.title("Precision-Recall Curve Comparison")
plt.xlabel("Recall")
plt.ylabel("Precision")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()